## Código fuente
### Agente Triaje y Conciliación de Medicamentos y Diagnóstico según ReAct

Este diseño permite que el sistema mantenga la **seguridad** de las reglas clínicas fijas mientras aprovecha la **flexibilidad** de la IA para conectar puntos de datos complejos en el diagnóstico.

A continuación ejecuta cada bloque de código

In [ ]:
import os
import json
from typing import List, Dict, Any, Callable

# Componentes de LangChain
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage

# =================================================================
# 1. CONFIGURACIÓN Y MODELO
# =================================================================
with open("content/clave_api.txt") as archivo:
    apikey = archivo.read().strip()

os.environ["OPENAI_API_KEY"] = apikey

# Definimos el modelo con temperatura 0 para precisión médica
model = ChatOpenAI(model="gpt-4o", temperature=0)

In [ ]:
# =================================================================
# 2. DEFINICIÓN DE HERRAMIENTAS (TOOLS)
# =================================================================

@tool
def triaje_determinista(temperatura: float, dolor_pecho: bool) -> str:
    """Ejecuta el protocolo de triaje basado en reglas fijas de salud."""
    if temperatura > 39.0 or dolor_pecho:
        return "PRIORIDAD ALTA: Derivación inmediata a sala de urgencias."
    return "PRIORIDAD MEDIA/BAJA: Evaluación estándar."

@tool
def consultar_vademecum(medicamento: str) -> str:
    """Busca contraindicaciones de un medicamento específico."""
    db = {"Warfarina": "Anticoagulante. Interacción crítica con AINEs.", 
          "Ibuprofeno": "No usar en gastritis crónica."}
    return db.get(medicamento, "Sin contraindicaciones graves registradas.")

@tool
def analizar_historial_clinico(paciente_id: str) -> str:
    """Consulta el expediente digital del paciente."""
    return f"Expediente {paciente_id}: Antecedentes de gastritis. Toma Warfarina."

# Registro y vinculación de herramientas
tools = [triaje_determinista, consultar_vademecum, analizar_historial_clinico]
model_with_tools = model.bind_tools(tools)

# Diccionario para mapear nombres de strings a funciones ejecutables
tool_map = {tool.name: tool 
            for tool in tools}

In [ ]:
# =================================================================
# 3. FUNCIONES AUXILIARES Y VALIDACIÓN
# =================================================================

def pre_raw_answer(ai_message: AIMessage) -> str:
    """
    Captura y formatea la respuesta operativa cruda del modelo 
    antes de la conciliación final.
    """
    content = ai_message.content
    if not content and ai_message.tool_calls:
        return "[Respuesta Técnica]: Generando llamadas a herramientas para validación clínica..."
    return content

def validar_y_ejecutar_herramienta(tool_call: Dict[str, Any]) -> ToolMessage:
    """Valida la existencia de la herramienta y la ejecuta de forma segura."""
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]
    
    if tool_name not in tool_map:
        return ToolMessage(
            tool_call_id=tool_call["id"],
            content=f"Error: La herramienta '{tool_name}' no existe."
        )
    
    # Ejecución dinámica
    print(f"--- Ejecutando herramienta: {tool_name} ---")
    resultado = tool_map[tool_name].invoke(tool_args)
    return ToolMessage(tool_call_id=tool_call["id"], content=str(resultado))

def mostrar_tool_calls(ai_msg: AIMessage):
    """Muestra de forma legible las llamadas a herramientas propuestas por el modelo."""
    if not ai_msg.tool_calls:
        print("El modelo no solicitó herramientas.")
        return
    
    for call in ai_msg.tool_calls:
        print(f"🔧 Tool Call Detectada: {call['name']}")
        print(f"📦 Argumentos: {json.dumps(call['args'], indent=2)}")

In [ ]:
# =================================================================
# 4. ORQUESTADOR DE CASO DE USO (Bucle de Ejecución)
# =================================================================

def ejecutar_caso_de_uso(consulta: str):
    """Bucle que permite al modelo razonar, usar herramientas y responder."""
    messages = [HumanMessage(content=consulta)]
    
    # Paso 1: El modelo razona y decide qué herramientas usar
    ai_msg = model_with_tools.invoke(messages)
    messages.append(ai_msg)

    # Captura de la respuesta operativa cruda inicial
    raw_obs = pre_raw_answer(ai_msg)
    print(f"\n🔍 OBSERVACIÓN PRE-PROCESADA: {raw_obs}")
    
    mostrar_tool_calls(ai_msg)
    
    # Paso 2: Si hay llamadas a herramientas, las ejecutamos
    if ai_msg.tool_calls:
        for tool_call in ai_msg.tool_calls:
            tool_msg = validar_y_ejecutar_herramienta(tool_call)
            messages.append(tool_msg)
        
        # Paso 3: El modelo genera la respuesta final con los datos obtenidos
        respuesta_final = model_with_tools.invoke(messages)
        print("\n✅ CONCLUSIÓN MÉDICA FINAL:")
        print(respuesta_final.content)
    else:
        print(ai_msg.content)

In [ ]:
# =================================================================
# 5. EJECUCIÓN
# =================================================================
if __name__ == "__main__":
    caso_clinico = """
    Paciente ID-701 llega con 35 de fiebre y presenta dolor de pecho. Se planea recetar Paracetamol.
    Valida el triaje y si el medicamento es seguro según su historial.
    """
    ejecutar_caso_de_uso(caso_clinico)

*Documentación elaborado por [Hadson Paredes](https://www.linkedin.com/in/hadson-paredes/) - 2026*
- Repositorio [Python-LangChain-Learning](https://github.com/devhadson/Python-LangChain-Learning/blob/main/lcel-Agente-ReAct-Determinista-y-Agentico/lcel-Agente-ReAct-Determinista-y-Agentico.ipynb)
- Disponible como curso en [Hadson.Tech](https://hadson.tech/cursos-disponibles/python-langChain)

<hr>
<h4 align="center"> Publicaciones en mis redes sociales y repositorio GitHub</h4>

<div align="center">
  <h3>Sígueme en mis redes sociales</h3>
  <a href="https://github.com/devhadson">
    <img src="https://img.shields.io/badge/GitHub-devhadson-black?logo=GitHub&style=flat-square" target="_blank" alt="GitHub">
  </a>
  <a href="https://www.linkedin.com/in/hadson-paredes/">
    <img src="https://img.shields.io/badge/LinkedIn-Hadson%20Paredes-blue?logo=linkedin&style=flat-square" target="_blank" alt="LinkedIn">
  </a>
  <a href="https://www.facebook.com/hadson.paredescordova/">
    <img src="https://img.shields.io/badge/Facebook-Hadson%20Paredes%20Cordova-Gree?logo=facebook&style=flat-square" target="_blank" alt="Facebook">
  </a>
  <a href="https://x.com/hadson_paredes">
    <img src="https://img.shields.io/badge/Hadson%20Paredes-black?logo=x&style=flat-square" target="_blank" alt="X">
  </a>
</div>